In [1]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode)


# Template Quest Notebook

In [2]:
import pandas as pd
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

## Part 1: Load the Sample Dataset

In [3]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/23 21:20:57 WARN Utils: Your hostname, Sams-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.20.6 instead (on interface en0)
26/05/23 21:20:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 21:20:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 1.1 NOTES 
- Read the data set using spark json auto parser.

In [4]:
# read dataset json file from relative path
data_file = 'data/sysmon_spearphish_cribl.json'
df_bronze = spark.read.json(data_file)

In [5]:
# making local copy of first three rows for a bit of manual exploration.
inspectionSample = df_bronze.take(3)
#print(inspectionSample[2])


#### 1.2 NOTES
- df_bronze is nested. "_raw" contains alert specific JSON.
- I need to flatten this into 2D.
- Issue: each eventID has unique fields in inner _raw JSON.
    - [x] Make a rough schema of fields for inner JSON, just useful looking fields.
        - This is a rough solution, a more robust method would be to pull an example of each eventID and extract/ iterate the fields for a comprehensive schema, but this solution serves for exploration, a comprehensive schema is not required at this stage -> min compute.
        - If task 2 requires further fields, or if I miss any I can revisit and parse necessary fields.
    - [x] Make a copy of bronze_df with another column containing _raw as parsed json from rough schema.
    - [x] select useful fields from parsed_raw as columns for silver_df.

In [6]:
# rough schema with fields to facilitate data exploration.
rough_schema = StructType([
    StructField("UtcTime", StringType(), True),
    StructField("Image", StringType(), True),
    StructField("ProcessId", StringType(), True),
    StructField("CommandLine", StringType(), True),
    StructField("ParentImage", StringType(), True),
    StructField("TargetFilename", StringType(), True),
    StructField("SourceImage", StringType(), True),
    StructField("TargetImage", StringType(), True),
    StructField("QueryName", StringType(), True),
    StructField("Hashes", StringType(), True),
])
# create a new bronze df with a column containing the inner JSON, parsed based on the rough schema.
df_bronze_parsed_raw = df_bronze.withColumn(
    "raw_parsed",
    from_json(col("_raw"), rough_schema)
)

In [7]:
df_bronze_parsed_raw.show(5)

+--------------------+-----------+---------+-------------------+--------------------+----------------+-------------+--------------------+------------+--------------------+--------------------+
|            Computer|Description|EventCode|               User|                _raw|           _time|cribl_breaker|          cribl_pipe|        host|              source|          raw_parsed|
+--------------------+-----------+---------+-------------------+--------------------+----------------+-------------+--------------------+------------+--------------------+--------------------+
|win-host-ctus-att...|       NULL|       23|NT AUTHORITY\SYSTEM|{"Name":"'Microso...|1.674818387593E9|     fallback|Splunk_UF_Windows...|f2c75b47cbb7|/tmp/cribldata/in...|{2023-01-27 11:19...|
|win-dc-ctus-attac...|       NULL|       23|NT AUTHORITY\SYSTEM|{"Name":"'Microso...|1.674818387419E9|     fallback|Splunk_UF_Windows...|f2c75b47cbb7|/tmp/cribldata/in...|{2023-01-27 11:19...|
|win-host-ctus-att...|       NULL| 

In [8]:
# make df_silver by selecting existing columns from df_bronze_parsed_raw and adding fields from the parsed inner JSON as their own columns.
df_silver = df_bronze_parsed_raw.select(
    #outer
    col("Computer"),
    col("EventCode"),
    col("User"),
    col("_time"),
    col("source"),
    col("host"),
    col("cribl_pipe"),
    col("host"),
    #inner
    col("raw_parsed.UtcTime").alias("UtcTime"),
    col("raw_parsed.Image").alias("Image"),
    col("raw_parsed.ProcessId").alias("ProcessId"),
    col("raw_parsed.CommandLine").alias("CommandLine"),
    col("raw_parsed.ParentImage").alias("ParentImage"),
    col("raw_parsed.TargetFilename").alias("TargetFilename"),
    col("raw_parsed.QueryName").alias("QueryName"),
    col("raw_parsed.SourceImage").alias("SourceImage"),
    col("raw_parsed.TargetImage").alias("TargetImage"),
    col("raw_parsed.Hashes").alias("Hashes")
    
)

In [9]:
df_silver.createOrReplaceTempView("sysmon_silver")

In [10]:
# Provided PySpark SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
LIMIT 5
""").show()

+--------------------+---------+-------------------+----------------+--------------------+------------+--------------------+------------+--------------------+--------------------+---------+-----------+-----------+--------------------+---------+--------------------+--------------------+--------------------+
|            Computer|EventCode|               User|           _time|              source|        host|          cribl_pipe|        host|             UtcTime|               Image|ProcessId|CommandLine|ParentImage|      TargetFilename|QueryName|         SourceImage|         TargetImage|              Hashes|
+--------------------+---------+-------------------+----------------+--------------------+------------+--------------------+------------+--------------------+--------------------+---------+-----------+-----------+--------------------+---------+--------------------+--------------------+--------------------+
|win-host-ctus-att...|       23|NT AUTHORITY\SYSTEM|1.674818387593E9|/tmp/cr

## Part 2: Detection Engineering

### Hypothesis
- *Adversaries may send spearphishing emails with a malicious attachment in an attempt to gain access to victim systems.*
- *A threat actor can leverage **Microsoft Office applications** against users on Windows endpoints through phishing to make **potentially malicious web calls**.*
- ***Windows Endpoints forwarding DNS activity via sysmon logs** to a data orchestration platform like Cribl would be utilized to identify this activity.*

Preferably using either PySpark or SQL, create a query or queries that will prove the detection hypothesis.
- [x] Output the result of this query in the notebook with a new dataframe
- [x] Select columns based on what is applicable to an analyst during triage



#### 2.1 NOTES
- Need to identify windows applications on windows endpoints forwarding DNS.
- This is found via sysmon eventID 22, *'DNS Queries made by any process on a Windows machine'*
   - As mentioned above, each event code has different field names, generic schema made above might leave out notable fields/ telemetry.
   - to avoid this I need to:
       - [x] extract _raw field from df_bronze for all eventID 22 logs,
       - [x] using one example, extract and iterate through field names,
       - [x] use these parsed fields to build a new, more robust schema.

In [11]:
#Make view of bronze df with column containing parsed JSON
df_bronze_parsed_raw.createOrReplaceTempView("sysmon_bronze_parsed_raw")

In [12]:
#collect parsed JSON of eventID 22 rows locally 
allRawFieldsEventID22 = spark.sql("""
SELECT _raw
FROM sysmon_bronze_parsed_raw
WHERE EventCode = 22
""").collect()

In [13]:
def schemaBuilder(rawJSON):
# takes JSON string, extracts field names, returns strucutred schema with those field names.
    fields= []
    parsed = json.loads(rawJSON)
    for key in parsed.keys():
        fields.append(StructField(key, StringType(), True))
    return StructType(fields)


In [14]:
# create schema of eventID 22 events using schemaBuilder function
eventId22Schema = schemaBuilder(allRawFieldsEventID22[0]["_raw"])

In [15]:
# create new dataframe with created schema, includes all fields relevant to eventID 22.
df_betterBronze = df_bronze.withColumn(
    "raw_parsed",
    from_json(col("_raw"), eventId22Schema) 
)

In [16]:
# flatten df_betterBronze by adding nested JSON columns as outer columns in new data frame, as done in part 1. 
df_silver = df_betterBronze.select(
    #outer
    col("Computer"),
    col("EventCode"),
    col("User"),
    col("_time"),
    col("source"),
    col("host"),
    col("cribl_pipe"),
    col("host"),
    #inner
    *[col(f"raw_parsed.{field.name}").alias(field.name) for field in eventId22Schema]
)



In [18]:
# verify
df_silver.columns

['Computer',
 'EventCode',
 'User',
 '_time',
 'source',
 'host',
 'cribl_pipe',
 'host',
 'Name',
 'Guid',
 'Version',
 'Level',
 'Task',
 'Opcode',
 'Keywords',
 'SystemTime',
 'EventRecordID',
 'ProcessID',
 'ThreadID',
 'Channel',
 'UserID',
 'RuleName',
 'UtcTime',
 'ProcessGuid',
 'ProcessId',
 'QueryName',
 'QueryStatus',
 'QueryResults',
 'Image']

In [19]:
#replace sysmon_silver view with with flattened dataframe, tuned for DNS alerting.
df_silver.createOrReplaceTempView("sysmon_silver")

26/05/23 21:21:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


#### 2.1 NOTES
- Now that I have this better dataframe, use SQL to prove hypothesis.
- Key characteristics of hypothesis include **DNS traffic** (eventID) from **windows applications** (Image) to **unexpected destinations** (QueryName)


In [20]:
# DNS sysmon Events
df_part2_output = spark.sql("""
SELECT UtcTime, Computer, EventCode, host, Name, Keywords, ProcessID, UserID, QueryName, QueryStatus, QueryResults, Image
FROM sysmon_silver
WHERE 
    EventCode = 22
AND Image IS NOT NULL
AND (
    Image LIKE '%Office%'
    OR Image LIKE '%WINWORD%'
    OR Image LIKE '%EXCEL%'
    OR Image LIKE '%POWERPNT%'
)
AND QueryName IS NOT NULL
AND QueryName NOT LIKE '%microsoft.com%'
AND QueryName NOT LIKE '%windows.com%'
AND QueryName NOT LIKE '%office.com%'
AND QueryName NOT LIKE '%office.net%'
AND QueryName NOT LIKE '%msedge.net%'
""")

In [21]:
df_part2_output.show()

+--------------------+--------------------+---------+------------+--------------------+------------------+---------+--------+-----------------+-----------+--------------------+--------------------+
|             UtcTime|            Computer|EventCode|        host|                Name|          Keywords|ProcessID|  UserID|        QueryName|QueryStatus|        QueryResults|               Image|
+--------------------+--------------------+---------+------------+--------------------+------------------+---------+--------+-----------------+-----------+--------------------+--------------------+
|2023-01-27 11:30:...|win-host-ctus-att...|       22|f2c75b47cbb7|'Microsoft-Window...|0x8000000000000000|   '1912'|S-1-5-18|www.mediafire.com|          0|::ffff:104.16.54....|C:\Program Files\...|
+--------------------+--------------------+---------+------------+--------------------+------------------+---------+--------+-----------------+-----------+--------------------+--------------------+



## Part 3: Additional Steps

I will use the ECS Framework.

I will use an LLM to map the existing field names as per recommendation from elastic official blog -> https://cribl.io/blog/the-easy-button-get-cribl-edge-windows-events-into-elastic/. The ideal scenario would be to configure in the cribl pipline but without access this is the next best solution.

- 'Computer' -> 'host.name'
- 'EventCode' -> 'event.code'
- 'User' -> 'user.name'
- '_time' -> '@timestamp'
- 'source' -> 'log.source'
- 'host' -> 'observer.hostname'
- 'cribl_pipe' -> 'observer.ingest.pipeline'
- 'Name' -> 'event.provider'
- 'Guid' -> 'event.provider_guid'
- 'Version' -> 'event.version'
- 'Level' -> 'log.level'
- 'Task' -> 'event.code'
- 'Opcode' -> 'event.action'
- 'Keywords' -> 'event.keywords'
- 'SystemTime' -> 'event.created'
- 'EventRecordID' -> 'event.sequence'
- 'Channel' -> 'log.logger'
- 'UserID' -> 'user.id'
- 'RuleName' -> 'rule.name'
- 'UtcTime' -> '@timestamp'  (preferred canonical time field)
- 'ProcessGuid' -> 'process.entity_id'
- 'ProcessId' -> 'process.pid'
- 'ThreadID' -> 'process.thread.id'
- 'QueryName' -> 'dns.question.name'
- 'QueryStatus' -> 'dns.response_code'
- 'QueryResults' -> 'dns.answers.data'
- 'Image' -> 'process.executable'


### Part 3.1: Normalization

In [22]:
# normalizing to ecs framework by aliasign columns.
df_silver_ecs = df_silver.select(
    col("Computer").alias("host.name"),
    col("host").alias("observer.hostname"),
    col("source").alias("log.source"),
    col("cribl_pipe").alias("observer.ingest.pipeline"),
    col("EventCode").alias("event.code"),
    col("Name").alias("event.provider"),
    col("Guid").alias("event.provider_guid"),
    col("Version").alias("event.version"),
    col("Level").alias("log.level"),
    col("Opcode").alias("event.action"),
    col("Keywords").alias("event.keywords"),
    col("EventRecordID").alias("event.sequence"),
    col("Channel").alias("log.logger"),
    col("RuleName").alias("rule.name"),
    col("UtcTime").alias("@timestamp"), # using Utctimestamp over _time as former is event time and latter is ingestion time, less relevant.
    col("User").alias("user.name"),
    col("UserID").alias("user.id"),
    col("Image").alias("process.executable"),
    col("ProcessId").alias("process.pid"),
    col("ProcessGuid").alias("process.entity_id"),
    col("ThreadID").alias("process.thread.id"),
    col("QueryName").alias("dns.question.name"),
    col("QueryStatus").alias("dns.response_code"),
    col("QueryResults").alias("dns.answers.data")
)

In [23]:
df_silver_ecs.createOrReplaceTempView("sysmon_GOLD")

### Part 3.2: Alert Table

#### 3.2.1 NOTES
Alert table will include metadata to aid the analyst and alert details as indicated below.

**alert metadata**
 - alert.title -> invented
 - alert.id -> invented
 - alet.description -> invented
 - alert.generated.timestamp -> invented
 - severity -> invented
 - mitre_technique - > T1566.001 (Spearphishing Attachment)

**alert details**
 - @timestamp
 - host.name
 - user.name
 - user.id
 - process.executable
 - process.pid
 - process.entity_id
 - process.thread.id
 - dns.question.name
 - dns.response.code
 - dns.answers.data


   

In [24]:
# collecting the suspicious event via sql.
triggering_event = spark.sql("""
SELECT *
FROM sysmon_GOLD
WHERE 
    `event.code` = 22
AND `process.executable` IS NOT NULL
AND (
    `process.executable` LIKE '%Office%'
    OR `process.executable` LIKE '%WINWORD%'
    OR `process.executable` LIKE '%EXCEL%'
    OR `process.executable` LIKE '%POWERPNT%'
)
AND `dns.question.name` IS NOT NULL
AND `dns.question.name` NOT LIKE '%microsoft.com%'
AND `dns.question.name` NOT LIKE '%windows.com%'
AND `dns.question.name` NOT LIKE '%office.com%'
AND `dns.question.name` NOT LIKE '%office.net%'
AND `dns.question.name` NOT LIKE '%msedge.net%'
""").collect()[0]

In [25]:
#manually building simulation alert table with invented metadata and details from the sql output.

values = [(
    '12345',
    'Possible Spearphishing Attachment - DNS Query',
    'A Windows executable has initiated DNS resolution for an potentially malicious domain.\n This may indicate a malicious phishing attachment attempting command and control (C2), payload retrieval or data exfiltration.\n Investigate the process tree, process hash, resolution, domain reputation and other similar DNS requests for evidence of beaconing.',
    '2026-05-22 10:17:51.000Z',
    'High',
    'T1566.001',
    '',
    str(triggering_event["@timestamp"]),
    str(triggering_event["event.code"]),
    str(triggering_event["host.name"]),
    str(triggering_event["observer.hostname"]),
    str(triggering_event["user.name"]),
    str(triggering_event["user.id"]),
    str(triggering_event["process.executable"]),
    str(triggering_event["process.pid"]),
    str(triggering_event["process.thread.id"]),
    str(triggering_event["process.entity_id"]),
    str(triggering_event["dns.question.name"]),
    str(triggering_event["dns.response_code"]),
    str(triggering_event["dns.answers.data"]),
)]  

keys = [
    "alert.id",
    "alert.title",
    "alert.description",
    "alert.generatedTime",
    "alert.severity",
    "alert.MITREMapping",
    "alert.linkedAlerts",
    "event.code",
    "@timestamp",
    "host.name",
    "observer.hostname",
    "user.name",
    "user.id",
    "process.executable",
    "process.pid",
    "process.thread.id",
    "process.entity_id",
    "dns.question.name",
    "dns.response_code",
    "dns.answers.data"
]

df_alert_table = spark.createDataFrame(values,keys)


In [26]:
# Final Output
df_alert_table.show()

+--------+--------------------+--------------------+--------------------+--------------+------------------+------------------+--------------------+----------+--------------------+-----------------+---------+--------+--------------------+-----------+-----------------+--------------------+-----------------+-----------------+--------------------+
|alert.id|         alert.title|   alert.description| alert.generatedTime|alert.severity|alert.MITREMapping|alert.linkedAlerts|          event.code|@timestamp|           host.name|observer.hostname|user.name| user.id|  process.executable|process.pid|process.thread.id|   process.entity_id|dns.question.name|dns.response_code|    dns.answers.data|
+--------+--------------------+--------------------+--------------------+--------------+------------------+------------------+--------------------+----------+--------------------+-----------------+---------+--------+--------------------+-----------+-----------------+--------------------+-----------------+--

### Part 3.3: Enrichment

### Virus Total:
- Domain www.Mediafire.come shows a community score of 0/100.
- notable community comment: '*MediaFire is a file hosting domain. While this website is often abused, it is legitimate.
Be aware that malicious advertisements and popups have been seen on MediaFire.
#abuse #legitimate #filehosting*'

### AbuseIPDB
- IP's ISP is Cloudflare, benign.
- IP was involved in Phishing Attachment IOC - according to source ArcSight Threat Intelligence - 2 years ago.

### Hybrid Analysis:
- Scan of the domain returns threat score of 100/100.
- vendor results:
    - urlscan.io | 0 malicious reports
    - scamAdviser | 0 malicious reports
    - CleanDNS | 271 malicious reports
    - BforeAI | 0 malicious reports
    - Criminal IP | 0 malicious reports
    - Vipre | 0 malicious reports
- Falcon Sandbox Historic Reports
    - 23/05/2023 | Threat Score 100/100 - Malicious
    - 21/12/2020 | Threat Score 80/100 - Malicious
    - 12/01/2025 | Threat Score 0/100 - No specific threat
    - 05/02/2019 | Threat Score 0/100 - No specific threat

### Comments
- TI indicates that the domain is a benign filesharing service via a cloudflare IP.
- The service itself is not malicious, but by nature of its file sharing function presents the threat of payload delivery/ data exfil.

## Summary

### Part 1
- I read in the data and parsed it into a dataframe using a rough schema I built through quick exploration of the log fields.
- Since each eventID had unique fields, I held off parsing all fields until I knew the eventID I would be looking for to avoid excessive null columns.
- This dataframe was only intended for exploration, I acknowledged the need to build a more robust schema when the search became more defined.

### Part 2
- I created a function to parse and build a schema for a given event ID.
- I used this function to create a new silver dataframe with columns tuned to searching DNS logs.
- I used spark sql to query the logs for DNS traffic from windows applications to unexpected destinations.

### Part 3
- I normalised the data in alignment with the ECS framework.
- I designed a simulation alert table containing data from the identified suspicious event, as well as invented metadata as I would expect an analyst to use in a SIEM.
- I used VirusTotal, AbuseIPDB and Hybrid Analysis to enrich alert for a prosective analyst.